# Transformer

**Domain:** Architectures  ·  **from study list**  ·  **runnable:** yes

The architecture from *Attention Is All You Need* (Vaswani et al., 2017) — the backbone of
modern LLMs, ViT, and most sequence models. This notebook builds the core pieces in plain
NumPy so the math is visible, then shows the real Hugging Face call shape.

## 1. What & Why

The **Transformer** throws out recurrence and convolution and models sequences with **attention
only**. Before it, sequence models were RNNs/LSTMs that process tokens one step at a time. That
has two costs: training can't be parallelized across the sequence (step *t* needs step *t-1*),
and information has to survive a long chain of updates, so distant tokens influence each other
only weakly.

Attention fixes both. Every token looks at **every other token in a single matrix multiply**, so
the path length between any two positions is *O(1)* and the whole sequence is processed in
parallel. That parallelism is exactly what made training on internet-scale data practical — it is
the reason GPT, BERT, T5, ViT, and friends all exist.

**Reach for it** for almost any sequence or set modeling problem where you have enough data and
compute: language, code, audio, images-as-patches ([[vision-transformer]]), proteins.
**Be careful** when sequences are very long — vanilla attention is *O(n²)* in time and memory —
or when data is tiny, where a simpler model (or a CNN/RNN inductive bias) will generalize
better.

## 2. Mental Model

**Attention is a soft, content-addressable dictionary lookup.**

Each token emits three vectors:

- a **query** *q* — "what am I looking for?"
- a **key** *k* — "what do I contain?"
- a **value** *v* — "what will I hand over if you pick me?"

A token's new representation is a **weighted average of all values**, where the weights come from
how well its query matches each key (dot product → softmax). Unlike a hash map that returns one
exact row, you get a *blend* of every row weighted by relevance.

```
            queries          keys                weights (softmax)        values     output
   token i:   q_i    ·   [k_1 k_2 ... k_n]   →   [w_1 w_2 ... w_n]   ·   [v_1..v_n] = z_i
                             (similarity)            (sum to 1)            (blend)
```

Stack many such layers, give each token a sense of *position*, wrap every sublayer in a residual
+ normalization, and you have a Transformer. The encoder lets every token see everything
(bidirectional); the decoder hides the future with a causal mask so it can generate one token at
a time.

## 3. Key Concepts

- **Scaled dot-product attention.** `Attention(Q,K,V) = softmax(Q·Kᵀ / √d_k) · V`. The `√d_k`
  divisor keeps dot products from growing with dimension and pushing softmax into a saturated,
  near-one-hot regime with vanishing gradients.

- **Multi-head attention.** Instead of one attention over the full `d_model`, split into `h`
  heads of size `d_model/h`, attend in each subspace independently, then concat and project. Heads
  specialize — one tracks syntax, another coreference, etc.

- **Positional encoding.** Attention is **permutation-invariant**: shuffle the tokens and you get
  the same (shuffled) outputs. To inject order you *add* a position signal — original paper uses
  fixed **sinusoids** of varying frequency; most LLMs now use learned or rotary (RoPE) positions.

- **Position-wise feed-forward (FFN).** A small MLP (`Linear → activation → Linear`, usually 4×
  wider in the middle) applied to each position independently. This is where most parameters and
  much of the "thinking" live.

- **Residuals + LayerNorm.** Every sublayer is `x + Sublayer(x)`, normalized. Residuals give
  gradients a clean highway; norm keeps activations well-scaled so you can stack dozens of layers.

- **Encoder vs decoder.** *Encoder* block = bidirectional self-attn + FFN. *Decoder* block =
  **causal** (masked) self-attn + **cross-attention** into the encoder output + FFN. Encoder-only
  → [[bert]]; decoder-only → GPT; both → [[t5]] / translation.

- **Causal mask.** In a decoder, set the upper triangle of the score matrix to `-∞` *before*
  softmax so position *t* can only attend to positions `≤ t` — no peeking at the future.

- **Pre-norm vs post-norm.** The 2017 paper puts LayerNorm *after* the residual add (post-norm);
  modern large models put it *before* the sublayer (pre-norm) for more stable deep training.

## 4. Setup

Examples 1–2 are **pure NumPy** and run on CPU in well under a second — no model downloads, so the
math is fully visible. Example 3 shows the real Hugging Face `transformers` call and is gated
behind an environment variable so the notebook still executes offline.

```bash
# only needed for the gated Example 3
%pip install transformers torch
```

In [1]:
import numpy as np

rng = np.random.default_rng(0)
np.set_printoptions(precision=3, suppress=True)
print("numpy", np.__version__)


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)      # stabilize
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)


numpy 2.5.0


## 5. Worked Examples

### Example 1 — Scaled dot-product attention, and what the causal mask does

We implement the core equation directly and inspect the **attention weight matrix**. Two things to
notice: every row sums to 1 (it's a probability distribution over the tokens being attended to),
and applying a causal mask zeroes out the upper triangle so each token only sees itself and the
past.

In [2]:
def attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)              # (seq, seq) similarity
    if mask is not None:
        scores = np.where(mask, scores, -np.inf)  # block disallowed positions
    weights = softmax(scores, axis=-1)
    return weights @ V, weights


seq, d_k = 4, 8
Q = rng.standard_normal((seq, d_k))
K = rng.standard_normal((seq, d_k))
V = rng.standard_normal((seq, d_k))

_, w_full = attention(Q, K, V)
print("bidirectional weights (rows sum to 1):")
print(w_full, "\nrow sums:", w_full.sum(axis=-1))

causal = np.tril(np.ones((seq, seq), dtype=bool))   # True where attending is allowed
_, w_causal = attention(Q, K, V, mask=causal)
print("\ncausal weights (upper triangle is zero):")
print(w_causal)


bidirectional weights (rows sum to 1):
[[0.302 0.472 0.081 0.145]
 [0.329 0.069 0.247 0.354]
 [0.32  0.388 0.233 0.059]
 [0.102 0.031 0.684 0.183]] 
row sums: [1. 1. 1. 1.]

causal weights (upper triangle is zero):
[[1.    0.    0.    0.   ]
 [0.827 0.173 0.    0.   ]
 [0.34  0.413 0.247 0.   ]
 [0.102 0.031 0.684 0.183]]


### Example 2 — A full encoder block, plus *proof* that you need positional encoding

Here we assemble multi-head attention, a feed-forward network, residuals, and LayerNorm into one
encoder block and confirm it preserves the `(seq, d_model)` shape. Then we demonstrate the
permutation-invariance trap: **without** position information, shuffling the input rows just
shuffles the output rows identically — the block has no idea what order the tokens came in. Adding
a positional encoding breaks that symmetry.

In [3]:
def layer_norm(x, eps=1e-5):
    mu = x.mean(-1, keepdims=True)
    var = x.var(-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps)


def multi_head_attention(X, Wq, Wk, Wv, Wo, n_heads):
    seq, d_model = X.shape
    d_head = d_model // n_heads
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    heads = []
    for h in range(n_heads):                      # split into per-head subspaces
        s = slice(h * d_head, (h + 1) * d_head)
        out, _ = attention(Q[:, s], K[:, s], V[:, s])
        heads.append(out)
    return np.concatenate(heads, axis=-1) @ Wo    # concat heads, project back


def encoder_block(X, params, n_heads):
    attn = multi_head_attention(X, *params["mha"], n_heads)
    X = layer_norm(X + attn)                       # residual + norm
    ff = np.maximum(X @ params["W1"], 0) @ params["W2"]   # FFN with ReLU
    return layer_norm(X + ff)                      # residual + norm


seq, d_model, n_heads, d_ff = 5, 16, 4, 64
def randn(*shape):
    return rng.standard_normal(shape) * 0.1

params = {
    "mha": [randn(d_model, d_model) for _ in range(4)],
    "W1": randn(d_model, d_ff),
    "W2": randn(d_ff, d_model),
}

X = rng.standard_normal((seq, d_model))
out = encoder_block(X, params, n_heads)
print("input shape ", X.shape, "-> output shape", out.shape)

# Permutation invariance: shuffle rows, compare outputs.
perm = rng.permutation(seq)
out_shuffled = encoder_block(X[perm], params, n_heads)
print("\nwithout positions, shuffling input just shuffles output? ",
      np.allclose(out_shuffled, out[perm]))

# Add sinusoidal positional encoding, then it is NOT invariant anymore.
pos = np.arange(seq)[:, None]
i = np.arange(d_model)[None, :]
angle = pos / np.power(10000, (i - i % 2) / d_model)
pe = np.where(i % 2 == 0, np.sin(angle), np.cos(angle))
out_pe = encoder_block(X + pe, params, n_heads)
out_pe_shuffled = encoder_block(X[perm] + pe, params, n_heads)
print("with positions, order now matters (outputs differ)?   ",
      not np.allclose(out_pe_shuffled, out_pe[perm]))


input shape  (5, 16) -> output shape (5, 16)

without positions, shuffling input just shuffles output?  True
with positions, order now matters (outputs differ)?    True


### Example 3 — A real pretrained Transformer (gated)

The NumPy code above *is* the architecture; production models just add scale, training, and
optimized kernels. This cell runs the real thing via Hugging Face. It downloads weights the first
time, so it only executes when `RUN_HF=1`; the call shape is printed either way.

In [4]:
import os

if os.getenv("RUN_HF"):
    from transformers import pipeline
    generator = pipeline("text-generation", model="distilgpt2")   # decoder-only Transformer
    out = generator("The Transformer architecture is", max_new_tokens=20,
                    do_sample=False)
    print(out[0]["generated_text"])
else:
    print("Set RUN_HF=1 to download a small Transformer (distilgpt2, ~350MB) and generate text.")
    print("Call shape:")
    print("  from transformers import pipeline")
    print("  gen = pipeline('text-generation', model='distilgpt2')")
    print("  gen('The Transformer architecture is', max_new_tokens=20)")


Set RUN_HF=1 to download a small Transformer (distilgpt2, ~350MB) and generate text.
Call shape:
  from transformers import pipeline
  gen = pipeline('text-generation', model='distilgpt2')
  gen('The Transformer architecture is', max_new_tokens=20)


## 6. Gotchas & Pitfalls

- **Quadratic cost.** Attention builds an `n×n` score matrix, so compute and memory grow with the
  *square* of sequence length. Long context needs FlashAttention (same math, IO-aware kernel) or a
  sparse/linear variant — see *When to Use* below.

- **Forgetting the `√d_k` scale.** Drop it and dot products grow with dimension, softmax saturates
  to near one-hot, and gradients through it vanish. This is a classic from-scratch bug.

- **No positional encoding = bag of words.** As Example 2 shows, attention is permutation
  invariant. If your model ignores word order, you almost certainly forgot positions.

- **Mask applied after softmax.** The causal/padding mask must set scores to `-∞` *before* softmax.
  Zeroing weights *after* softmax leaves the distribution unnormalized and leaks future information.

- **Causal vs padding mask.** They're different: the **causal** mask hides the future (decoder
  only); the **padding** mask hides `[PAD]` tokens (both). Real models combine both.

- **Post-norm deep stacks are fragile.** The original post-norm design needs LR warmup and careful
  init to train deep; pre-norm is the modern default for stability.

- **`d_model` must divide by `n_heads`.** Each head gets `d_model / n_heads` dimensions — a
  non-integer split is a silent shape bug.

## 7. When to Use vs Alternatives

| Option | Pick it when… | Trade-off vs vanilla Transformer |
|---|---|---|
| **Transformer (full attention)** | Default for sequence/set modeling with enough data & compute | *O(n²)* cost; weak inductive bias needs lots of data |
| **RNN / LSTM / GRU** | Streaming, very long or unbounded sequences, tiny data | Sequential (no parallel training); weak long-range memory |
| **CNN (e.g. TCN)** | Local patterns, low latency, modest sequence length | Limited receptive field; needs depth/dilation for long range |
| **FlashAttention** | You want full attention but it's memory-bound | Same math/results, just a faster IO-aware kernel — almost always use it |
| **Sparse / linear attention** (Longformer, Performer) | Very long sequences where *O(n²)* is infeasible | Approximate or restricted attention; some quality loss |
| **State-space models** ([[mamba-ssm]]) | Extremely long sequences, linear-time inference | Newer, less tooling; different training dynamics |

In practice: start with a standard Transformer + FlashAttention. Only move to a sparse/linear or
SSM variant once sequence length makes quadratic attention the actual bottleneck.

## 8. Resources

- **Attention Is All You Need** — Vaswani et al., 2017, the original paper:
  https://arxiv.org/abs/1706.03762
- **The Illustrated Transformer** — Jay Alammar's visual walkthrough:
  https://jalammar.github.io/illustrated-transformer/
- **The Annotated Transformer** — Harvard NLP's line-by-line PyTorch implementation:
  https://nlp.seas.harvard.edu/annotated-transformer/
- **Hugging Face Transformers docs** — models, tokenizers, and the `pipeline` API:
  https://huggingface.co/docs/transformers/index
- Related notebooks here: [[bert]] (encoder-only), [[t5]] (encoder-decoder),
  [[vision-transformer]] (images as patches), [[attention-mechanisms]] (attention zoo).